In [ ]:
# 1. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# 2. 필요한 패키지 설치
!pip install --upgrade transformers datasets accelerate --quiet

In [ ]:
# 3. 주요 라이브러리 로딩
import json
import torch
from datasets import Dataset
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [ ]:
# 4. 모델 및 토크나이저 로딩
model_name = "skt/kogpt2-base-v2"
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.add_special_tokens({'eos_token': '</s>', 'pad_token': '<pad>'})
model.resize_token_embeddings(len(tokenizer))

In [ ]:
# 5. 문제 데이터 로딩
from google.colab import files
import json

uploaded = files.upload()

with open("augmented_math.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
# 6. 정답생성용 프롬프트
formatted_data = []
for item in data:
    q = item["Question"]
    a = item["Answer"]
    full_text = f"문제: {q}\n정답: {a}"
    formatted_data.append({"Text": full_text, "Question": f"문제: {q}\n", "Answer": f"정답: {a}"})

In [ ]:
# 7. 마스킹 토크나이즈 함수
def tokenize_with_mask(example):
    full_input = tokenizer(example["Text"], padding="max_length", truncation=True, max_length=512)
    question_input = tokenizer(example["Question"], padding="max_length", truncation=True, max_length=512)

    labels = full_input["input_ids"][:]
    q_len = sum([1 for id in question_input["input_ids"] if id != tokenizer.pad_token_id])

    # 질문 부분 마스킹
    labels[:q_len] = [-100] * q_len
    full_input["labels"] = labels

    return full_input

In [ ]:
# 8. Dataset 변환 및 전처리
dataset = Dataset.from_list(formatted_data)
tokenized_dataset = dataset.map(tokenize_with_mask, batched=False)

In [ ]:
# 9. Trainer 준비
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/KoGPT/CheckPoint/Math/Answer",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=1,
    logging_steps=10,
    eval_strategy="no",
    fp16=torch.cuda.is_available(),
    overwrite_output_dir=True
)

In [ ]:
# 10. Trainer 구성
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

In [ ]:
# 11. 학습 실행
trainer.train()

In [ ]:
# 12. 모델 저장
model.save_pretrained("/content/drive/MyDrive/Colab Notebooks/KoGPT/Model/Math/Answer")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/KoGPT/Model/Math/Answer")